# Data build from WRDS

Builds the enriched SDC file (`data/interim/sdc_pr_52w_high_decile.csv`) and the acquirer return files (`data/interim/acquirer_decile_*.csv`) that `02_analysis.ipynb` reads.
Requires a WRDS account with access to **SDC Mergers & Acquisitions** and **CRSP** (`crsp.dsf_v2`, `crsp.stocknames`), and `pip install wrds`. This notebook is a record of the build and is not run as part of the analysis.
Each step adds columns only; the original SDC columns are never changed, and duplicate deal rows are flagged, not deleted.

> **Known gaps.** Three steps happened outside this notebook:
> 1. loading the CRSP `stocknames` table into `names`;
> 2. the tie-breaker that resolves deals with several candidate PERMNOs and creates `tpermno_final` / `apermno_final`;
> 3. the pull of `pctacq` / `pctown` (the query below now includes them; the analysis reads them from `data/raw/sdc_us_public_2005_2025_pct.csv`).
>
> The enriched file in `data/interim/` already contains their output.

In [ ]:
import numpy as np
import pandas as pd
import wrds

KEYS = ["master_cusip", "acusip", "dateann"]   # one deal = target CUSIP + acquirer CUSIP + announcement date

In [ ]:
SDC_QUERY = """
    SELECT
        tr_sdc_ma.wrds_ma_details.master_cusip,
        tr_sdc_ma.wrds_ma_details.dateann,
        tr_sdc_ma.wrds_ma_details.amv,
        tr_sdc_ma.wrds_ma_details.dateannorig,
        tr_sdc_ma.wrds_ma_details.status,
        tr_sdc_ma.wrds_ma_details.acusip,
        tr_sdc_ma.wrds_ma_details.aup,
        tr_sdc_ma.wrds_ma_details.tup,
        tr_sdc_ma.wrds_ma_details.anation,
        tr_sdc_ma.wrds_ma_details.tnation,
        tr_sdc_ma.wrds_ma_details.tpublic,
        tr_sdc_ma.wrds_ma_details.apublic,
        tr_sdc_ma.wrds_ma_details.tticker,
        tr_sdc_ma.wrds_ma_details.deal_value,
        tr_sdc_ma.wrds_ma_details.pr,
        tr_sdc_ma.wrds_ma_details.consid,
        tr_sdc_ma.wrds_ma_details.considoff,
        tr_sdc_ma.wrds_ma_details.pctacq,
        tr_sdc_ma.wrds_ma_details.pctown
    FROM tr_sdc_ma.wrds_ma_details
    WHERE tr_sdc_ma.wrds_ma_details.dateann BETWEEN
        '2005-01-01'::date AND '2025-12-31'::date
    AND ("tr_sdc_ma"."wrds_ma_details"."tnation" = 'United States'
         AND "tr_sdc_ma"."wrds_ma_details"."anation" = 'United States')
    AND tpublic = 'Public'
    AND apublic = 'Public'
    AND status = 'Completed'
"""

In [ ]:
db = wrds.Connection()
sdc = db.raw_sql(SDC_QUERY)
sdc.shape

## 1. Map target and acquirer to CRSP PERMNOs
Join on 6-character CUSIP within the name-validity window; fall back to the ultimate parent's CUSIP when the direct match fails.

In [ ]:
def add_join_keys(sdc, names):
    """Add working columns only. Nothing original is changed or removed.

    Adds 6-character CUSIP keys for the join and a datetime copy of the announcement date
    for the validity-window filter. dateann itself keeps its exact string values.
    """
    sdc["dateann_dt"] = pd.to_datetime(sdc["dateann"])
    sdc["tcusip6"] = sdc["master_cusip"].astype(str).str[:6]  # target
    sdc["acusip6"] = sdc["acusip"].astype(str).str[:6]        # acquirer
    sdc["tup6"] = sdc["tup"].astype(str).str[:6]              # target ultimate parent
    sdc["aup6"] = sdc["aup"].astype(str).str[:6]              # acquirer ultimate parent
    names["cusip6"] = names["cusip"].astype(str).str[:6]
    return sdc, names

def flag_duplicates(sdc):
    """Add a flag for duplicate deal rows. No rows are removed.

    A duplicate is a repeated (target CUSIP, acquirer CUSIP, announcement date) key. Every row
    sharing a key that appears more than once is flagged, so the duplicates stay visible.
    """
    sdc["is_dup"] = sdc.duplicated(subset=KEYS, keep=False)
    return sdc

def candidates(sdc, names, cusip_col):
    """Return the (deal, PERMNO) candidate table for one side.

    Joins the deals to CRSP stocknames on the 6-character CUSIP and keeps only name records whose
    validity window (namedt to nameenddt) covers the announcement date. A deal with several share
    classes appears once per candidate PERMNO. The original frames are not touched.
    """
    m = sdc.merge(names, left_on=cusip_col, right_on="cusip6", how="left")
    in_window = (m["namedt"] <= m["dateann_dt"]) & (m["dateann_dt"] <= m["nameenddt"])
    return m.loc[in_window, KEYS + ["permno"]]

def attach_permno(sdc, cand, permno_col, src_col, multi_col, source):
    """Add the chosen PERMNO and its flags as new columns.

    Picks the first candidate PERMNO per deal, records the source, and flags deals with several
    candidates so the tie-breaker can refine them later. Only adds columns.
    """
    first = cand.drop_duplicates(subset=KEYS, keep="first")[KEYS + ["permno"]]
    n_cand = cand.groupby(KEYS, as_index=False)["permno"].nunique()
    n_cand = n_cand.rename(columns={"permno": "n_cand"})
    out = sdc.merge(first, on=KEYS, how="left")
    out = out.merge(n_cand, on=KEYS, how="left")
    out = out.rename(columns={"permno": permno_col})
    out[multi_col] = out["n_cand"].fillna(0) > 1
    out[src_col] = np.where(out[permno_col].notna(), source, "none")
    return out.drop(columns=["n_cand"])

def attach_fallback(sdc, cand, permno_col, src_col, parent_label="parent"):
    """Fill the still-empty PERMNO slots from the parent-CUSIP candidates.

    Only rows whose PERMNO column is empty are filled; the source is set to parent for filled rows.
    """
    first = cand.drop_duplicates(subset=KEYS, keep="first")[KEYS + ["permno"]]
    first = first.rename(columns={"permno": "permno_fb"})
    out = sdc.merge(first, on=KEYS, how="left")
    fill = out[permno_col].isna() & out["permno_fb"].notna()
    out.loc[fill, permno_col] = out.loc[fill, "permno_fb"]
    out.loc[fill, src_col] = parent_label
    return out.drop(columns=["permno_fb"])

def report_mapping(sdc):
    """Return the mapping counts in distinct deals."""
    dedup = sdc.drop_duplicates(subset=KEYS)
    both = dedup["tpermno"].notna() & dedup["apermno"].notna()
    return {
        "distinct_deals": len(dedup),
        "target_mapped": int(dedup["tpermno"].notna().sum()),
        "target_multi": int(dedup["tpermno_multi"].sum()),
        "target_via_parent": int((dedup["tpermno_src"] == "parent").sum()),
        "acquirer_mapped": int(dedup["apermno"].notna().sum()),
        "acquirer_multi": int(dedup["apermno_multi"].sum()),
        "acquirer_via_parent": int((dedup["apermno_src"] == "parent").sum()),
        "both_mapped": int(both.sum()),
    }

def add_is_matched(sdc):
    """Add is_matched = 'y' when both the target and the acquirer have a final PERMNO, else 'n'."""
    out = sdc.copy()
    out["is_matched"] = "n"
    out.loc[out["tpermno_final"].notna() & out["apermno_final"].notna(), "is_matched"] = "y"
    return out

In [ ]:
# names = <CRSP stocknames: permno, cusip, namedt, nameenddt>   <- load step not in this notebook (see note above)
sdc, names = add_join_keys(sdc, names)
sdc = flag_duplicates(sdc)

sdc = attach_permno(sdc, candidates(sdc, names, "tcusip6"), "tpermno", "tpermno_src", "tpermno_multi", "direct")
sdc = attach_fallback(sdc, candidates(sdc, names, "tup6"), "tpermno", "tpermno_src", "parent")
sdc = attach_permno(sdc, candidates(sdc, names, "acusip6"), "apermno", "apermno_src", "apermno_multi", "direct")
sdc = attach_fallback(sdc, candidates(sdc, names, "aup6"), "apermno", "apermno_src", "parent")
report_mapping(sdc)

## 2. Tie-breaker and matched flag
*Tie-breaker step not in this notebook:* it resolves deals with `tpermno_multi` / `apermno_multi` and creates `tpermno_final` and `apermno_final`.

In [ ]:
# sdc = <tie-breaker>   <- creates tpermno_final / apermno_final
sdc = add_is_matched(sdc)
(sdc.drop_duplicates(subset=KEYS)["is_matched"] == "y").sum()

## 3. Target 52-week high
Maximum daily price over [dateann - 400, dateann - 21] calendar days, pulled from CRSP one year at a time.

In [ ]:
def pull_by_year(db, windows, permno_col, columns):
    """Pull ``columns`` from crsp.dsf_v2 in one query per calendar year.

    For each year, only the PERMNOs whose window overlaps that year are requested.
    Returns the concatenated rows with ``dlycaldt`` parsed as a date.
    """
    years = range(windows["win_start"].dt.year.min(), windows["win_end"].dt.year.max() + 1)
    frames = []
    for y in years:
        y_start, y_end = pd.Timestamp(f"{y}-01-01"), pd.Timestamp(f"{y}-12-31")
        overlap = (windows["win_start"] <= y_end) & (windows["win_end"] >= y_start)
        permnos = windows.loc[overlap, permno_col].unique()
        if len(permnos) == 0:
            continue
        plist = ",".join(str(int(p)) for p in sorted(set(int(x) for x in permnos)))
        frames.append(db.raw_sql(f"""
            SELECT permno, dlycaldt, {columns}
            FROM crsp.dsf_v2
            WHERE permno IN ({plist})
            AND dlycaldt BETWEEN '{y}-01-01'::date AND '{y}-12-31'::date
        """))
    out = pd.concat(frames, ignore_index=True)
    out["dlycaldt"] = pd.to_datetime(out["dlycaldt"])
    return out


def build_target_windows(sdc):
    """One pull window per distinct deal with a mapped target: dateann - 400 days to dateann - 21 days."""
    dedup = sdc.drop_duplicates(subset=KEYS)
    mapped = dedup[dedup["tpermno_final"].notna()].copy()
    mapped["win_start"] = mapped["dateann_dt"] - pd.Timedelta(days=400)
    mapped["win_end"] = mapped["dateann_dt"] - pd.Timedelta(days=21)
    return mapped[KEYS + ["tpermno_final", "dateann_dt", "win_start", "win_end"]]

def pull_target_prices(db, windows):
    """Pull target daily prices (dlyprc), one query per calendar year."""
    return pull_by_year(db, windows, "tpermno_final", "dlyprc")

def compute_52wk_high(windows, prices):
    """52-week high for each deal window: the maximum daily price inside the window.

    Deals with no prices in the window get NaN.
    """
    by_permno = {p: g.sort_values("dlycaldt").reset_index(drop=True) for p, g in prices.groupby("permno")}
    highs = []
    for _, w in windows.iterrows():
        series = by_permno.get(int(w["tpermno_final"]))
        if series is None or len(series) == 0:
            highs.append(float("nan"))
            continue
        window = series[(series["dlycaldt"] >= w["win_start"]) & (series["dlycaldt"] <= w["win_end"])]
        highs.append(window["dlyprc"].max() if len(window) else float("nan"))
    out = windows[KEYS + ["tpermno_final"]].copy()
    out["target_52wk_high"] = highs
    return out

def add_target_52wk_high(sdc, highs):
    """Add the target 52-week high as a new column, dropping any stale copy from an earlier run first."""
    out = sdc.copy()
    stale = [c for c in out.columns if c == "target_52wk_high" or c.startswith("target_52wk_high_")]
    out = out.drop(columns=stale)
    return out.merge(highs[KEYS + ["target_52wk_high"]], on=KEYS, how="left")

In [ ]:
windows = build_target_windows(sdc)
prices = pull_target_prices(db, windows)
highs = compute_52wk_high(windows, prices)
sdc = add_target_52wk_high(sdc, highs)

## 4. Offer price where SDC has none
`pr_filled = deal_value x 1000 / shrout` (last CRSP shares outstanding on or before the announcement). This assumes `deal_value` is the price of 100% of the target's equity; see the limitations in `docs/METHODS.md`.

In [ ]:
def build_fill_windows(sdc):
    """One shares-outstanding window (dateann - 400 days to dateann) per matched deal that has deal_value but no pr."""
    dedup = sdc.drop_duplicates(subset=KEYS)
    fill = dedup[(dedup["is_matched"] == "y") & dedup["pr"].isna() & dedup["deal_value"].notna()].copy()
    fill["win_start"] = fill["dateann_dt"] - pd.Timedelta(days=400)
    fill["win_end"] = fill["dateann_dt"]
    return fill[KEYS + ["tpermno_final", "dateann_dt", "deal_value", "win_start", "win_end"]]

def pull_target_shrout(db, windows):
    """Pull target shares outstanding (shrout), one query per calendar year."""
    return pull_by_year(db, windows, "tpermno_final", "shrout")

def compute_pr_fill(windows, shrout):
    """Filled offer price: last shares outstanding on or before the announcement, pr = deal_value * 1000 / shrout.

    deal_value is in $ millions and CRSP shrout is in thousands of shares.
    NOTE: this assumes deal_value is the price of 100% of the target's equity. It is not for
    self-deals, buybacks or partial-stake purchases; see docs/METHODS.md.
    """
    by_permno = {p: g.sort_values("dlycaldt").reset_index(drop=True) for p, g in shrout.groupby("permno")}
    filled = []
    for _, w in windows.iterrows():
        series = by_permno.get(int(w["tpermno_final"]))
        if series is None or len(series) == 0:
            filled.append(float("nan"))
            continue
        before = series[series["dlycaldt"] <= w["dateann_dt"]]
        if len(before) == 0:
            filled.append(float("nan"))
            continue
        last_shrout = before.iloc[-1]["shrout"]
        filled.append(float("nan") if pd.isna(last_shrout) or last_shrout == 0
                      else w["deal_value"] * 1000 / last_shrout)
    out = windows[KEYS + ["tpermno_final"]].copy()
    out["pr_filled"] = filled
    return out

def add_pr_filled(sdc, filled):
    """Add the filled offer price as a new column, dropping any stale copy from an earlier run first."""
    out = sdc.copy()
    stale = [c for c in out.columns if c == "pr_filled" or c.startswith("pr_filled_")]
    out = out.drop(columns=stale)
    return out.merge(filled[KEYS + ["pr_filled"]], on=KEYS, how="left")

In [ ]:
fill_windows = build_fill_windows(sdc)
shrout = pull_target_shrout(db, fill_windows)
sdc = add_pr_filled(sdc, compute_pr_fill(fill_windows, shrout))

## 5. AnchorDistance

In [ ]:
def add_anchor_distance(sdc):
    """Add offer_price (pr where present, else pr_filled) and anchor_distance = offer_price / 52-week high - 1."""
    out = sdc.copy()
    out["offer_price"] = out["pr"].fillna(out["pr_filled"])
    out["anchor_distance"] = out["offer_price"] / out["target_52wk_high"] - 1
    return out

def add_initial_anchor_decile(sdc):
    """First-pass decile on all matched deals with an AnchorDistance (decile 1 = lowest).

    The analysis later re-forms the deciles on the third-party, 100%-owned sample
    (see the analysis notebook).
    """
    out = sdc.copy()
    out["anchor_decile"] = float("nan")
    mask = (out["is_matched"] == "y") & out["anchor_distance"].notna()
    out.loc[mask, "anchor_decile"] = pd.qcut(out.loc[mask, "anchor_distance"], 10, labels=False) + 1
    return out

In [ ]:
sdc = add_anchor_distance(sdc)
sdc = add_initial_anchor_decile(sdc)   # first-pass decile; the analysis notebook re-forms it on the final sample

## 6. Acquirer daily returns
[dateann - 400, dateann + 400] calendar days per matched deal, split into ten files by decile.

In [ ]:
def build_acquirer_windows(sdc):
    """One pull window per matched deal for the acquirer: dateann - 400 days to dateann + 400 days.

    The 400 days before cover the market-model estimation window (A-252 to A-31 trading days);
    the 400 days after cover the 12-month holding period.
    """
    dedup = sdc.drop_duplicates(subset=KEYS)
    matched = dedup[dedup["is_matched"] == "y"].copy()
    matched["win_start"] = matched["dateann_dt"] - pd.Timedelta(days=400)
    matched["win_end"] = matched["dateann_dt"] + pd.Timedelta(days=400)
    return matched[KEYS + ["apermno_final", "anchor_decile", "dateann_dt", "win_start", "win_end"]]

def pull_acquirer_panel(db, windows):
    """Pull acquirer daily prices, returns, volume, shares and market cap, one query per calendar year."""
    return pull_by_year(db, windows, "apermno_final", "dlyprc, dlyret, dlyvol, shrout, dlycap, siccd")

def split_by_decile(windows, panel):
    """Split the acquirer panel into one frame per decile (dict decile -> frame).

    Cuts each deal's window from its own PERMNO series and tags it with the deal key and decile.
    Deals without a decile are skipped.
    """
    by_permno = {p: g.sort_values("dlycaldt").reset_index(drop=True) for p, g in panel.groupby("permno")}
    buckets = {d: [] for d in range(1, 11)}
    for _, w in windows.iterrows():
        if pd.isna(w["anchor_decile"]):
            continue
        series = by_permno.get(int(w["apermno_final"]))
        if series is None or len(series) == 0:
            continue
        window = series[(series["dlycaldt"] >= w["win_start"]) & (series["dlycaldt"] <= w["win_end"])].copy()
        if len(window) == 0:
            continue
        for k in KEYS:
            window[k] = w[k]
        window["anchor_decile"] = int(w["anchor_decile"])
        buckets[int(w["anchor_decile"])].append(window)
    return {d: pd.concat(buckets[d], ignore_index=True) if buckets[d] else pd.DataFrame() for d in range(1, 11)}

In [ ]:
acq_windows = build_acquirer_windows(sdc)
panel = pull_acquirer_panel(db, acq_windows)
by_decile = split_by_decile(acq_windows, panel)
for d, frame in by_decile.items():
    frame.to_csv(f"../data/interim/acquirer_decile_{d}.csv", index=False)

## 7. Save the enriched SDC file

In [ ]:
# Write PERMNO columns as integers so their text format is unchanged.
# sdc.to_csv("../data/interim/sdc_pr_52w_high_decile.csv", index=False)